# 泰坦尼克号乘客生存预测分析

姓名：周贤中

## 1. 课题背景

泰坦尼克号沉船事件是数据分析和机器学习入门中最常见的案例之一。该数据集同时包含数值型特征、分类型特征以及缺失值问题，适合用于演示完整的数据分析流程：数据获取、数据清洗、探索性分析、特征工程、分类建模和结果解释。

本报告围绕“哪些因素会影响乘客生存，以及能否基于乘客信息预测其是否生还”这一问题展开，最终完成了可复现的 Python 分析脚本，并输出可视化图表和分类模型评估结果。

## 2. 研究目标

本次课程报告的目标如下：

1. 对泰坦尼克号乘客数据进行清洗和整理。
2. 从性别、舱位等级、年龄等角度分析生存率差异。
3. 构建乘客生存预测模型，并评估模型效果。
4. 形成一份可复现、可运行、可解释的 Python 数据分析报告。

## 3. 数据来源与字段说明

- 数据来源：seaborn Titanic 公开数据集，已缓存到 report/data/titanic.csv。
- 样本量：511 条乘客记录。
- 目标变量：survived，1 表示生还，0 表示未生还。
- 主要特征：

| 字段 | 含义 |
| --- | --- |
| pclass | 舱位等级，1 为头等舱，3 为三等舱 |
| sex | 性别 |
| age | 年龄 |
| sibsp | 同行兄弟姐妹或配偶数量 |
| parch | 同行父母或子女数量 |
| fare | 船票价格 |
| embarked | 登船港口 |

在原始字段基础上，脚本额外构造了两个特征：

- family_size = sibsp + parch + 1
- is_alone：是否独自出行

## 4. 分析环境

- Python 3.12
- pandas
- matplotlib
- seaborn
- scikit-learn

完整可运行代码见 report/titanic_analysis.py。

## 5. 数据清洗与特征工程

清洗阶段主要完成了以下工作：

1. 保留建模需要的核心字段。
2. 按性别和舱位等级对年龄缺失值进行分组中位数填补。
3. 用众数填补 embarked，用中位数填补 fare。
4. 构造 family_size 和 is_alone 两个衍生特征。
5. 使用年龄分段，便于后续探索分析。

In [7]:
from __future__ import annotations

import json
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


DATA_DIR = Path("data")
FIGURE_DIR = Path("figures")
OUTPUT_DIR = Path("output")
DATA_FILE = DATA_DIR / "titanic.csv"

DATA_URLS = [
    "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv",
    "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv",
]


def ensure_directories() -> None:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def ensure_dataset() -> Path:
    if DATA_FILE.exists():
        return DATA_FILE

    for url in DATA_URLS:
        try:
            urlretrieve(url, DATA_FILE)
            return DATA_FILE
        except URLError:
            continue

    raise RuntimeError(
        "Unable to download Titanic dataset. Check network access or place titanic.csv in report/data/."
    )


def load_dataset() -> pd.DataFrame:
    dataset_path = ensure_dataset()
    df = pd.read_csv(dataset_path)

    if "survived" not in df.columns and "Survived" in df.columns:
        df = df.rename(
            columns={
                "Survived": "survived",
                "Pclass": "pclass",
                "Sex": "sex",
                "Age": "age",
                "SibSp": "sibsp",
                "Parch": "parch",
                "Fare": "fare",
                "Embarked": "embarked",
            }
        )

    keep_columns = ["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
    missing_columns = [column for column in keep_columns if column not in df.columns]
    if missing_columns:
        raise ValueError(f"Dataset is missing required columns: {missing_columns}")

    return df[keep_columns].copy()


def clean_and_engineer(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()

    age_group_median = cleaned.groupby(["sex", "pclass"])["age"].transform("median")
    cleaned["age"] = cleaned["age"].fillna(age_group_median)
    cleaned["age"] = cleaned["age"].fillna(cleaned["age"].median())
    cleaned["embarked"] = cleaned["embarked"].fillna(cleaned["embarked"].mode().iloc[0])
    cleaned["fare"] = cleaned["fare"].fillna(cleaned["fare"].median())

    cleaned["family_size"] = cleaned["sibsp"] + cleaned["parch"] + 1
    cleaned["is_alone"] = (cleaned["family_size"] == 1).astype(int)
    cleaned["age_band"] = pd.cut(
        cleaned["age"],
        bins=[0, 12, 18, 35, 60, 100],
        labels=["child", "teen", "young_adult", "adult", "senior"],
        include_lowest=True,
    )

    return cleaned


def save_dataset_profile(df: pd.DataFrame) -> dict:
    survival_by_sex = (df.groupby("sex")["survived"].mean().sort_values(ascending=False) * 100).round(2)
    survival_by_class = (df.groupby("pclass")["survived"].mean().sort_index() * 100).round(2)
    survival_by_age = (df.groupby("age_band", observed=False)["survived"].mean() * 100).round(2)

    profile = {
        "sample_count": int(len(df)),
        "survival_rate": round(float(df["survived"].mean() * 100), 2),
        "female_survival_rate": round(float(survival_by_sex.get("female", 0.0)), 2),
        "male_survival_rate": round(float(survival_by_sex.get("male", 0.0)), 2),
        "first_class_survival_rate": round(float(survival_by_class.get(1, 0.0)), 2),
        "third_class_survival_rate": round(float(survival_by_class.get(3, 0.0)), 2),
        "young_adult_survival_rate": round(float(survival_by_age.get("young_adult", 0.0)), 2),
        "adult_survival_rate": round(float(survival_by_age.get("adult", 0.0)), 2),
    }

    with (OUTPUT_DIR / "dataset_profile.json").open("w", encoding="utf-8") as file:
        json.dump(profile, file, indent=2)

    return profile


def create_figures(df: pd.DataFrame) -> None:
    sns.set_theme(style="whitegrid", palette="deep")

    plt.figure(figsize=(7, 5))
    ax = sns.barplot(data=df, x="sex", y="survived", estimator="mean", errorbar=None)
    ax.set_title("Survival Rate by Sex")
    ax.set_xlabel("Sex")
    ax.set_ylabel("Survival Rate")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "survival_by_sex.png", dpi=180)
    plt.close()

    plt.figure(figsize=(7, 5))
    ax = sns.barplot(data=df, x="pclass", y="survived", estimator="mean", errorbar=None)
    ax.set_title("Survival Rate by Passenger Class")
    ax.set_xlabel("Passenger Class")
    ax.set_ylabel("Survival Rate")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "survival_by_class.png", dpi=180)
    plt.close()

    plt.figure(figsize=(8, 5))
    sns.histplot(data=df, x="age", hue="survived", bins=25, kde=True, element="step", stat="density", common_norm=False)
    plt.title("Age Distribution by Survival")
    plt.xlabel("Age")
    plt.ylabel("Density")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "age_distribution_by_survival.png", dpi=180)
    plt.close()

    corr_df = df[["survived", "pclass", "age", "sibsp", "parch", "fare", "family_size", "is_alone"]].corr(numeric_only=True)
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr_df, annot=True, fmt=".2f", cmap="Blues", square=True)
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "correlation_heatmap.png", dpi=180)
    plt.close()


def build_model_data(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series, list[str], list[str]]:
    feature_columns = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "family_size", "is_alone"]
    X = df[feature_columns].copy()
    y = df["survived"].astype(int)
    numeric_features = ["age", "sibsp", "parch", "fare", "family_size", "is_alone", "pclass"]
    categorical_features = ["sex", "embarked"]
    return X, y, numeric_features, categorical_features


def evaluate_models(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, str]:
    X, y, numeric_features, categorical_features = build_model_data(df)

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                numeric_features,
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]),
                categorical_features,
            ),
        ]
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    model_specs = {
        "logistic_regression": LogisticRegression(max_iter=1000, random_state=42),
        "random_forest": RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=4, random_state=42),
    }

    records = []
    best_name = ""
    best_auc = -1.0
    best_confusion = None

    for name, estimator in model_specs.items():
        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", estimator),
        ])
        pipeline.fit(X_train, y_train)
        predictions = pipeline.predict(X_test)
        probabilities = pipeline.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, probabilities)

        records.append(
            {
                "model": name,
                "accuracy": round(float(accuracy_score(y_test, predictions)), 4),
                "precision": round(float(precision_score(y_test, predictions)), 4),
                "recall": round(float(recall_score(y_test, predictions)), 4),
                "f1": round(float(f1_score(y_test, predictions)), 4),
                "roc_auc": round(float(auc), 4),
            }
        )

        if auc > best_auc:
            best_auc = auc
            best_name = name
            best_confusion = confusion_matrix(y_test, predictions)

    metrics_df = pd.DataFrame(records).sort_values(by="roc_auc", ascending=False).reset_index(drop=True)
    metrics_df.to_csv(OUTPUT_DIR / "model_metrics.csv", index=False)

    confusion_df = pd.DataFrame(
        best_confusion,
        index=["actual_0", "actual_1"],
        columns=["pred_0", "pred_1"],
    )
    confusion_df.to_csv(OUTPUT_DIR / "best_model_confusion_matrix.csv")

    return metrics_df, confusion_df, best_name


def create_confusion_matrix_figure(confusion_df: pd.DataFrame, best_model_name: str) -> None:
    plt.figure(figsize=(6, 5))
    sns.heatmap(confusion_df, annot=True, fmt="d", cmap="Greens")
    plt.title(f"Confusion Matrix of {best_model_name}")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "best_model_confusion_matrix.png", dpi=180)
    plt.close()


def create_markdown_summary(profile: dict, metrics_df: pd.DataFrame, best_model_name: str) -> None:
    best_row = metrics_df.loc[metrics_df["model"] == best_model_name].iloc[0]
    lines = [
        "# Titanic Analysis Summary",
        "",
        f"- Sample count: {profile['sample_count']}",
        f"- Overall survival rate: {profile['survival_rate']}%",
        f"- Female survival rate: {profile['female_survival_rate']}%",
        f"- Male survival rate: {profile['male_survival_rate']}%",
        f"- First class survival rate: {profile['first_class_survival_rate']}%",
        f"- Third class survival rate: {profile['third_class_survival_rate']}%",
        f"- Best model: {best_model_name}",
        f"- Accuracy: {best_row['accuracy']}",
        f"- Precision: {best_row['precision']}",
        f"- Recall: {best_row['recall']}",
        f"- F1: {best_row['f1']}",
        f"- ROC AUC: {best_row['roc_auc']}",
    ]

    with (OUTPUT_DIR / "analysis_summary.md").open("w", encoding="utf-8") as file:
        file.write("\n".join(lines))


def main() -> None:
    ensure_directories()
    raw_df = load_dataset()
    clean_df = clean_and_engineer(raw_df)

    profile = save_dataset_profile(clean_df)
    create_figures(clean_df)
    metrics_df, confusion_df, best_model_name = evaluate_models(clean_df)
    create_confusion_matrix_figure(confusion_df, best_model_name)
    create_markdown_summary(profile, metrics_df, best_model_name)

    print("Titanic analysis completed.")
    print(f"Dataset saved at: {DATA_FILE}")
    print(f"Best model: {best_model_name}")
    print(metrics_df.to_string(index=False))

if __name__ == "__main__":
    main()

Titanic analysis completed.
Dataset saved at: data/titanic.csv
Best model: random_forest
              model  accuracy  precision  recall     f1  roc_auc
      random_forest    0.8350     0.8966   0.650 0.7536   0.8200
logistic_regression    0.8155     0.8182   0.675 0.7397   0.7851


In [5]:
main()

Titanic analysis completed.
Dataset saved at: data/titanic.csv
Best model: random_forest
              model  accuracy  precision  recall     f1  roc_auc
      random_forest    0.8350     0.8966   0.650 0.7536   0.8200
logistic_regression    0.8155     0.8182   0.675 0.7397   0.7851


## 3. 数据来源与字段说明

- 数据来源：seaborn Titanic 公开数据集，已缓存到 report/data/titanic.csv。
- 样本量：511 条乘客记录。
- 目标变量：survived，1 表示生还，0 表示未生还。
- 主要特征：

| 字段 | 含义 |
| --- | --- |
| pclass | 舱位等级，1 为头等舱，3 为三等舱 |
| sex | 性别 |
| age | 年龄 |
| sibsp | 同行兄弟姐妹或配偶数量 |
| parch | 同行父母或子女数量 |
| fare | 船票价格 |
| embarked | 登船港口 |

在原始字段基础上，脚本额外构造了两个特征：

- family_size = sibsp + parch + 1
- is_alone：是否独自出行

## 4. 分析环境

- Python 3.12
- pandas
- matplotlib
- seaborn
- scikit-learn

完整可运行代码见 report/titanic_analysis.py。

## 5. 数据清洗与特征工程

清洗阶段主要完成了以下工作：

1. 保留建模需要的核心字段。
2. 按性别和舱位等级对年龄缺失值进行分组中位数填补。
3. 用众数填补 embarked，用中位数填补 fare。
4. 构造 family_size 和 is_alone 两个衍生特征。
5. 使用年龄分段，便于后续探索分析。

关键代码如下：

```python
def clean_and_engineer(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()

    age_group_median = cleaned.groupby(["sex", "pclass"])["age"].transform("median")
    cleaned["age"] = cleaned["age"].fillna(age_group_median)
    cleaned["age"] = cleaned["age"].fillna(cleaned["age"].median())
    cleaned["embarked"] = cleaned["embarked"].fillna(cleaned["embarked"].mode().iloc[0])
    cleaned["fare"] = cleaned["fare"].fillna(cleaned["fare"].median())

    cleaned["family_size"] = cleaned["sibsp"] + cleaned["parch"] + 1
    cleaned["is_alone"] = (cleaned["family_size"] == 1).astype(int)
    return cleaned
```

这种处理方式兼顾了数据完整性与可解释性。相比直接删除缺失值，分组填补更适合本案例中年龄受舱位和性别共同影响的特点。

## 6. 探索性分析

### 6.1 总体生存情况

- 总体生存率：38.75%
- 女性生存率：74.21%
- 男性生存率：17.76%
- 头等舱生存率：57.14%
- 三等舱生存率：27.97%

从整体数据看，生存者并非多数，说明这是一个类别分布并不完全均衡的二分类任务。

### 6.2 性别对生存率的影响

![按性别划分的生存率](figures/survival_by_sex.png)

图中可以明显看出，女性生存率远高于男性。结合历史背景，这与当时“妇女和儿童优先”的救援原则一致。性别是本数据中最显著的解释变量之一。

### 6.3 舱位等级对生存率的影响

![按舱位等级划分的生存率](figures/survival_by_class.png)

头等舱乘客的生存率明显高于三等舱乘客，说明社会地位、舱位位置和获得救援机会之间可能存在较强关联。舱位等级越高，生存概率越高。

### 6.4 年龄分布与生存情况

![按生存与否划分的年龄分布](figures/age_distribution_by_survival.png)

年龄分布显示，儿童和年轻乘客的生存概率相对更高，中老年乘客生存优势不明显。根据脚本统计，青年组生存率为 38.16%，成年组为 35.09%，说明年龄确实有一定影响，但影响程度低于性别和舱位等级。

### 6.5 数值变量相关性

![相关性热力图](figures/correlation_heatmap.png)

相关性热力图表明：

1. pclass 与 survived 呈负相关，说明舱位等级数字越大，生存率越低。
2. fare 与 survived 呈正相关，船票价格更高的乘客往往更容易生还。
3. family_size 与是否独自出行之间存在明显关系，说明家庭同行情况可能影响逃生机会。

## 7. 建模方法

本报告将问题定义为二分类任务，并比较了两种常见模型：

1. 逻辑回归：作为基线模型，优点是结构简单、解释性较强。
2. 随机森林：能够处理非线性关系和特征交互，适合提高预测表现。

建模流程如下：

1. 选择特征 pclass、sex、age、sibsp、parch、fare、embarked、family_size、is_alone。
2. 将数值变量做缺失填补和标准化。
3. 将分类变量做众数填补和独热编码。
4. 按 8:2 划分训练集与测试集。
5. 以准确率、精确率、召回率、F1 和 ROC AUC 对模型进行评估。

关键建模代码如下：

```python
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ]
)

model_specs = {
    "logistic_regression": LogisticRegression(max_iter=1000, random_state=42),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=4,
        random_state=42,
    ),
}
```

## 8. 模型结果与分析

模型评估结果如下：

| 模型 | Accuracy | Precision | Recall | F1 | ROC AUC |
| --- | ---: | ---: | ---: | ---: | ---: |
| Random Forest | 0.8350 | 0.8966 | 0.6500 | 0.7536 | 0.8200 |
| Logistic Regression | 0.8155 | 0.8182 | 0.6750 | 0.7397 | 0.7851 |

从结果看，随机森林的综合表现最好，尤其在准确率和 ROC AUC 上优于逻辑回归，因此本报告选择随机森林作为最终模型。

### 8.1 混淆矩阵分析

随机森林的混淆矩阵如下：

| 实际值\\预测值 | Pred 0 | Pred 1 |
| --- | ---: | ---: |
| Actual 0 | 60 | 3 |
| Actual 1 | 14 | 26 |

![最佳模型混淆矩阵](figures/best_model_confusion_matrix.png)

可以看出：

1. 模型对未生还乘客的识别较好，误判为生还的只有 3 人。
2. 模型对生还乘客的识别仍有遗漏，14 名实际生还者被预测为未生还。
3. 这也解释了为什么模型精确率较高，但召回率还有提升空间。

换言之，该模型在“预测某人会生还”时较为谨慎，因此预测为生还的样本通常较可信，但会漏掉一部分真实生还者。

## 9. 结论

通过本次分析，可以得到以下结论：

1. 性别是影响生存率的关键因素，女性生存率显著高于男性。
2. 舱位等级对生存有明显影响，头等舱乘客生存率高，三等舱乘客风险更大。
3. 年龄存在一定影响，但作用强度弱于性别和舱位等级。
4. 基于乘客基本信息构建的分类模型可以实现较好的预测效果，其中随机森林模型在测试集上达到 0.8350 的准确率和 0.8200 的 ROC AUC。

整体而言，泰坦尼克号乘客的生存并不是随机事件，而是与社会身份、资源条件和个体属性存在显著关联。

## 10. 局限性与改进方向

本报告仍存在以下局限：

1. 当前使用的特征较少，未加入船舱位置、票号分组等更细粒度信息。
2. 模型只做了基础参数设置，尚未进行系统调参。
3. 数据规模有限，模型结论更适合作为教学案例，不宜过度外推。

后续可以从以下方向改进：

1. 增加更多衍生特征，例如票价分层、家庭身份标签等。
2. 使用交叉验证和网格搜索进一步优化模型参数。
3. 增加特征重要性分析，提高模型解释性。

## 11. 复现方式

在项目根目录执行以下命令即可重新生成图表和模型结果：

```bash
/mnt/d/Github/python-data-analysis/.venv/bin/python report/titanic_analysis.py
```

脚本运行后会自动生成以下内容：

- report/data/titanic.csv
- report/figures/*.png
- report/output/*.csv
- report/output/analysis_summary.md

## 12. 附录

- 分析脚本：report/titanic_analysis.py
- 数据文件：report/data/titanic.csv
- 结果汇总：report/output/analysis_summary.md

本报告完成了一个完整的 Python 数据分析闭环，体现了从数据获取到模型评估再到结论表达的全过程，满足课程报告对 Markdown 主体与独立 Python 代码文件的要求。